# 01 - Build the text analysis data

This notebook starts from the **existing processed files**. It checks them, then runs `Research_Analysis/run_research_analysis.py`, which adds transparent text clues and saves student, session, problem, and conversation summaries. It also produces topic, clustering, and treatment-comparison outputs.

The Python script is the single source of the calculations; this notebook explains and displays each stage as it is.

**Arm labels:** `vanilla` = GPT Base; `aug` = GPT Tutor. T


## 1. Find the project and check the starting files

Run from the project folder or open this notebook from inside it.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "ProcessedDatasets").is_dir()
     and (p / "Research_Analysis" / "run_research_analysis.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository or set PROJECT_ROOT to the project folder.")

DATA_DIR = PROJECT_ROOT / "ProcessedDatasets"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "research"
required = [
    DATA_DIR / "cleaned_user_messages.csv",
    DATA_DIR / "student_features_final.csv",
    DATA_DIR / "preprocessing_config.json",
    PROJECT_ROOT / "PreProcessing" / "conversations_audit.csv",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Required project inputs are missing:\n" + "\n".join(missing))
print(f"Project: {PROJECT_ROOT}")
print(f"New generated files: {OUTPUT_DIR}")


Project: C:\Users\SrijanWork\Documents\Coding\Data-MIning-Project
New generated files: C:\Users\SrijanWork\Documents\Coding\Data-MIning-Project\outputs\research


## 2. Confirm the old files and configuration

The configuration is read from disk so the existing feature definitions remain the source of truth.

In [2]:
config = json.loads((DATA_DIR / "preprocessing_config.json").read_text(encoding="utf-8"))
students = pd.read_csv(DATA_DIR / "student_features_final.csv")
messages = pd.read_csv(DATA_DIR / "cleaned_user_messages.csv", low_memory=False)

print(f"Student table: {students.shape[0]:,} rows ? {students.shape[1]} columns")
print(f"Student messages: {messages.shape[0]:,} rows ? {messages.shape[1]} columns")
print(f"Unique students: {students['username'].nunique():,}; duplicate student IDs: {students['username'].duplicated().sum()}")
print("Treatment labels:", students["treatment"].value_counts().to_dict())
print("Configured primary features:", config["primary_features"])
print("Configured supplementary rates:", config["supplementary_rate_features"])
print("Configured supplementary flags:", config["supplementary_binary_features"])
assert students["username"].is_unique
assert set(students["treatment"].dropna().unique()) == {"aug", "vanilla"}


Student table: 518 rows ? 33 columns
Student messages: 28,664 rows ? 23 columns
Unique students: 518; duplicate student IDs: 0
Treatment labels: {'aug': 277, 'vanilla': 241}
Configured primary features: ['student_message_count', 'average_word_count', 'average_question_count', 'explanation_request_rate', 'complete_solution_request_rate', 'verification_request_rate', 'conceptual_question_rate', 'code_prompt_rate', 'followup_prompt_rate']
Configured supplementary rates: ['hint_request_rate', 'refinement_request_rate', 'multiple_question_rate']
Configured supplementary flags: ['hint_request_present', 'refinement_request_present', 'multiple_question_present']


## 3. See how one prompt becomes text clues

The next cell imports the same feature function used by the Python pipeline. It shows the real extraction code and applies it to one existing message. The full pipeline below applies this function to every prompt.

In [3]:
import inspect
import sys

sys.path.insert(0, str(PROJECT_ROOT / "Research_Analysis"))
from run_research_analysis import text_features

print(inspect.getsource(text_features))
example_prompt = messages["message"].iloc[0]
print("\nExample student prompt:", example_prompt)
display(pd.Series(text_features(example_prompt), name="extracted clue"))


def text_features(message: str) -> dict[str, float | int]:
    s = str(message)
    words = re.findall(r"\b[\w']+\b", s.lower(), flags=re.UNICODE)
    unique = len(set(words))
    sentences = len(re.findall(r"[.!?]+", s))
    math_terms = re.findall(r"\b(?:equation|factor|multiple|prime|integer|fraction|ratio|variable|coefficient|polynomial|function|probability|theorem|angle|area|perimeter|slope|matrix|derivative|sum|product|quotient|expression|solve|simplify)\w*\b", s.lower())
    out = {
        "lexical_diversity": unique / len(words) if words else 0.0,
        "mean_token_length": float(np.mean([len(w) for w in words])) if words else 0.0,
        "mean_words_per_sentence": len(words) / sentences if sentences else float(len(words)),
        "clause_marker_count": len(re.findall(r"\b(?:because|although|however|therefore|while|since|if|then)\b", s.lower())),
        "math_terminology_count": len(math_terms),
        "math_terminology_present": int(bool(math_terms)),
        "punctuati

lexical_diversity              1.0
mean_token_length              3.4
mean_words_per_sentence        5.0
clause_marker_count            0.0
math_terminology_count         0.0
math_terminology_present       0.0
punctuation_count              0.0
direct_answer_request          0.0
clarification_request          0.0
example_request                0.0
mistake_explanation_request    0.0
alternative_method_request     0.0
independent_attempt            0.0
reasoning_explanation          0.0
specific_difficulty            0.0
why_correct_request            0.0
approach_comparison            0.0
challenge_or_verification      0.0
builds_on_explanation          0.0
Name: extracted clue, dtype: float64

## 4. Build the derived text and interaction outputs

The pipeline adds interpretable text markers (for example, attempts, requests for hints, and mathematical terms), computes them before aggregation, then produces multi-level datasets. It also reads the separate full conversation export when analyzing actual assistant turns. No API or paid model is used.

In [4]:
script = PROJECT_ROOT / "Research_Analysis" / "run_research_analysis.py"
run = subprocess.run(
    [sys.executable, str(script)],
    cwd=PROJECT_ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print(run.stdout[-5000:])
if run.stderr.strip():
    print("\nScript messages (stderr):\n", run.stderr[-2000:])


{
  "seed": 42,
  "input_student_rows": 518,
  "input_student_unique": 518,
  "input_cleaned_student_message_rows": 28664,
  "student_message_role_values": [
    "user"
  ],
  "timestamp_missing_student_messages": 23215,
  "validated_student_username_join": true,
  "derived_rows": {
    "message_text_features": 28664,
    "students": 518,
    "student_session": 1771,
    "student_problem": 2484,
    "conversation_user_summary": 7089
  },
  "assistant_turn_audit": {
    "available": true,
    "rows": 64486,
    "role_counts": {
      "assistant": 28731,
      "user": 28666,
      "system": 7089
    },
    "assistant_rows": 28731,
    "assistant_followed_by_user_pairs": 21575,
    "assistant_response_categories": [
      "response_direct_answer",
      "response_hint_or_scaffold",
      "response_explanation",
      "response_guided_question"
    ],
    "assistant_response_flag_counts": {
      "response_direct_answer": 4348,
      "response_hint_or_scaffold": 7947,
      "response_expla

## 5. Check the saved datasets

Each row level answers a different question, so these are separate files rather than renamed copies: one row per prompt, student, session, problem, or conversation.

In [5]:
dataset_names = [
    "message_text_features.csv",
    "student_features_with_text.csv",
    "student_session_features.csv",
    "student_problem_features.csv",
    "conversation_student_features.csv",
]
rows = []
for name in dataset_names:
    path = OUTPUT_DIR / "datasets" / name
    frame = pd.read_csv(path, low_memory=False)
    rows.append({"file": name, "rows": len(frame), "columns": len(frame.columns)})
display(pd.DataFrame(rows))


,file,rows,columns
0,message_text_features.csv,28664,44
1,student_features_with_text.csv,518,57
2,student_session_features.csv,1771,44
3,student_problem_features.csv,2484,44
4,conversation_student_features.csv,7089,30


In [6]:
message_features = pd.read_csv(OUTPUT_DIR / "datasets" / "message_text_features.csv", nrows=8)
show = [c for c in ["username", "session_id", "problem_id", "message", "independent_attempt",
                    "reasoning_explanation", "hint_request", "math_terminology_count"]
        if c in message_features.columns]
display(message_features[show])


,username,session_id,problem_id,message,independent_attempt,reasoning_explanation,hint_request,math_terminology_count
0,181671,1,4,"""what is an even numbr""",0,0,0,0
1,181671,1,4,"""prme number is what""",0,0,0,0
2,130337,1,2,"""what is q""",0,0,0,0
3,130337,1,2,"""q is true""",0,0,0,0
4,161087,1,3,"""any idea how to go about solving this one""",0,0,0,0
5,161087,1,3,"""a[if]b[not]?""",0,0,0,0
6,161087,1,3,"""b[if](a[or]c)nb[if]c[not]""",0,0,0,0
7,161087,1,3,"""what,""",0,0,0,0


## 6. Review automated text labels

The sample below is a human-check sheet. Its automated labels are candidates, not ground truth; the blank review fields should be filled only after a person checks the original prompt.

In [7]:
review_path = OUTPUT_DIR / "tables" / "annotation_review_sample.csv"
review = pd.read_csv(review_path)
display(review.head(12))


,username,conversation_id,message,manual_review_label,reviewer_notes,direct_answer_request,clarification_request,example_request,mistake_explanation_request,alternative_method_request,independent_attempt,reasoning_explanation,specific_difficulty,why_correct_request,approach_comparison,challenge_or_verification,builds_on_explanation
0,161520,d71372d5-de72-4742-8247-b87d1cde1f14,hava nasu0131,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0
1,110608,66b2ca60-e171-4c3e-b1a7-b8d3f24b7b85,y=2mx,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0
2,110628,6b0be248-e9fe-4c64-af1a-89740822d1fe,n,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0
3,151122,30e29022-eb50-49fe-ba5f-775d73fb04ca,1/3,NaN,NaN,0,0,0,0,0,1,0,0,0,0,0,0
4,229796,c3e51e2d-ebb8-460a-b1de-73eaa9577258,3 and -5 ?,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0
5,130243,42e4b570-56da-4874-864c-107f821b4ae2,is the answer for b part is one prime number i...,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0
6,162955,2ae4698f-e17d-4c19-80f4-5a84cb286b36,what is the value of c,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0
7,110608,66b2ca60-e171-4c3e-b1a7-b8d3f24b7b85,how do i do tht,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0
8,171750,31af0e29-9e8d-4c43-a6b4-3c07ef750a18,n(A)-31?,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0
9,130167,4a9dd025-66ed-4d26-a2aa-1adf538dbbf3,so answe is a(ax+b=x,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0


## What this step created

The **original cleaned student messages** remain under `ProcessedDatasets/`. New engineered datasets, cluster/topic tables, charts, and run summary are under `outputs/research/`. Continue with [02 - Interaction profiles and treatment](02_interaction_profiles_and_treatment.ipynb).


## Questions you may be asked

- **What did you add?** We kept every original prompt and added visible text clues, such as requests for hints, attempts, explanation requests, and math words.
- **Why make several CSVs?** One file keeps one row per prompt; other files summarize the prompts for a student, session, problem, or conversation.
- **Do these clues prove that a student understood?** No. They record wording that can be checked; they do not directly measure understanding.
